# Fashion-MNIST Neural Network Comparison

**Author:** Marsya Putra  

This is a single executable notebook for a controlled Fashion-MNIST neural-network comparison project. It contains the complete workflow: environment setup, data loading, preprocessing checks, Dense Neural Network experiments, CNN experiments, optimizer/learning-rate comparison, ranking, visualizations, confusion matrix, and misclassification analysis.

The notebook writes CSV result files to `csv/`.


## 1. Setup and Environment

The runtime environment is recorded so the experiment can be interpreted and rerun more easily.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import Markdown, display
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

PROJECT_ROOT = Path.cwd()
CSV_DIR = PROJECT_ROOT / "csv"
CSV_DIR.mkdir(exist_ok=True)

# Source assignment documents are intentionally not included in the public portfolio repository.
# The project evidence in this notebook is the executed code, generated metrics, and CSV outputs.

runtime_info = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "gpu_devices": [device.name for device in tf.config.list_physical_devices("GPU")],
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
}
display(pd.DataFrame([runtime_info]))


## 2. Dataset Loading and Validation

Fashion-MNIST is loaded from Keras. The official test split is preserved for final evaluation. A stratified validation split is derived from the original training data.


In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE,
    random_state=SEED,
    stratify=y_train_full,
)

# Normalize pixels to [0, 1] and add channel dimension for CNN compatibility.
x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]
x_val = (x_val.astype("float32") / 255.0)[..., np.newaxis]
x_test = (x_test.astype("float32") / 255.0)[..., np.newaxis]

y_train = y_train.astype("int64")
y_val = y_val.astype("int64")
y_test = y_test.astype("int64")

assert x_train.shape[1:] == (28, 28, 1)
assert x_val.shape[1:] == (28, 28, 1)
assert x_test.shape[1:] == (28, 28, 1)
assert set(np.unique(y_train)) == set(range(10))
assert x_train.min() >= 0 and x_train.max() <= 1
assert x_val.min() >= 0 and x_val.max() <= 1
assert x_test.min() >= 0 and x_test.max() <= 1

split_df = pd.DataFrame([
    {"split": "train", "samples": len(x_train), "shape": str(tuple(x_train.shape)), "pixel_min": float(x_train.min()), "pixel_max": float(x_train.max())},
    {"split": "validation", "samples": len(x_val), "shape": str(tuple(x_val.shape)), "pixel_min": float(x_val.min()), "pixel_max": float(x_val.max())},
    {"split": "test", "samples": len(x_test), "shape": str(tuple(x_test.shape)), "pixel_min": float(x_test.min()), "pixel_max": float(x_test.max())},
])
display(split_df)

class_balance = []
for split_name, labels in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    values, counts = np.unique(labels, return_counts=True)
    for value, count in zip(values, counts):
        class_balance.append({"split": split_name, "class_id": int(value), "class_name": CLASS_NAMES[int(value)], "count": int(count)})
class_balance_df = pd.DataFrame(class_balance)
display(class_balance_df.pivot(index="class_name", columns="split", values="count"))

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=class_balance_df, x="class_name", y="count", hue="split", ax=ax)
ax.set_title("Fashion-MNIST Class Balance by Split")
ax.set_xlabel("Class")
ax.set_ylabel("Samples")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


## 3. Model Definitions

Task 1 uses Dense Neural Networks only. Task 2 uses Convolutional Neural Networks. Task 3 keeps the selected final CNN architecture fixed and compares optimizer or learning-rate settings.


In [ ]:
def build_model(architecture_key: str, seed: int = SEED) -> keras.Model:
    set_seed(seed)
    inputs = keras.Input(shape=(28, 28, 1), name="image")

    if architecture_key == "dnn_baseline":
        x = layers.Flatten(name="flatten")(inputs)
        x = layers.Dense(128, activation="relu", name="dense_128_relu")(x)
        outputs = layers.Dense(10, activation="softmax", name="class_output")(x)

    elif architecture_key == "dnn_deeper":
        x = layers.Flatten(name="flatten")(inputs)
        x = layers.Dense(256, activation="relu", name="dense_256_relu")(x)
        x = layers.Dense(128, activation="relu", name="dense_128_relu")(x)
        outputs = layers.Dense(10, activation="softmax", name="class_output")(x)

    elif architecture_key == "dnn_tuned":
        x = layers.Flatten(name="flatten")(inputs)
        x = layers.Dense(256, activation="relu", name="dense_256_relu")(x)
        x = layers.Dropout(0.25, name="dropout_025")(x)
        x = layers.Dense(128, activation="relu", name="dense_128_relu")(x)
        outputs = layers.Dense(10, activation="softmax", name="class_output")(x)

    elif architecture_key == "cnn_baseline":
        x = layers.Conv2D(32, 3, activation="relu", padding="same", name="conv_32")(inputs)
        x = layers.MaxPooling2D(name="pool_1")(x)
        x = layers.Conv2D(64, 3, activation="relu", padding="same", name="conv_64")(x)
        x = layers.MaxPooling2D(name="pool_2")(x)
        x = layers.Flatten(name="flatten")(x)
        x = layers.Dense(128, activation="relu", name="dense_128_relu")(x)
        outputs = layers.Dense(10, activation="softmax", name="class_output")(x)

    elif architecture_key == "cnn_more_filters":
        x = layers.Conv2D(64, 3, activation="relu", padding="same", name="conv_64_a")(inputs)
        x = layers.MaxPooling2D(name="pool_1")(x)
        x = layers.Conv2D(128, 3, activation="relu", padding="same", name="conv_128")(x)
        x = layers.MaxPooling2D(name="pool_2")(x)
        x = layers.Flatten(name="flatten")(x)
        x = layers.Dense(128, activation="relu", name="dense_128_relu")(x)
        outputs = layers.Dense(10, activation="softmax", name="class_output")(x)

    elif architecture_key == "cnn_deeper":
        x = layers.Conv2D(32, 3, activation="relu", padding="same", name="conv_32_a")(inputs)
        x = layers.Conv2D(32, 3, activation="relu", padding="same", name="conv_32_b")(x)
        x = layers.MaxPooling2D(name="pool_1")(x)
        x = layers.Conv2D(64, 3, activation="relu", padding="same", name="conv_64_a")(x)
        x = layers.Conv2D(64, 3, activation="relu", padding="same", name="conv_64_b")(x)
        x = layers.MaxPooling2D(name="pool_2")(x)
        x = layers.Flatten(name="flatten")(x)
        x = layers.Dense(64, activation="relu", name="dense_64_relu")(x)
        outputs = layers.Dense(10, activation="softmax", name="class_output")(x)

    elif architecture_key == "cnn_tuned_finalist":
        x = layers.Conv2D(32, 3, activation="relu", padding="same", name="conv_32_a")(inputs)
        x = layers.BatchNormalization(name="bn_1")(x)
        x = layers.Conv2D(32, 3, activation="relu", padding="same", name="conv_32_b")(x)
        x = layers.MaxPooling2D(name="pool_1")(x)
        x = layers.Dropout(0.20, name="dropout_020")(x)
        x = layers.Conv2D(64, 3, activation="relu", padding="same", name="conv_64_a")(x)
        x = layers.BatchNormalization(name="bn_2")(x)
        x = layers.Conv2D(64, 3, activation="relu", padding="same", name="conv_64_b")(x)
        x = layers.MaxPooling2D(name="pool_2")(x)
        x = layers.Dropout(0.30, name="dropout_030")(x)
        x = layers.Flatten(name="flatten")(x)
        x = layers.Dense(64, activation="relu", name="dense_64_relu")(x)
        outputs = layers.Dense(10, activation="softmax", name="class_output")(x)

    else:
        raise ValueError(f"Unknown architecture_key: {architecture_key}")

    return keras.Model(inputs=inputs, outputs=outputs, name=architecture_key)


def build_optimizer(name: str, learning_rate: float) -> keras.optimizers.Optimizer:
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "sgd_momentum":
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    if name == "rmsprop":
        return keras.optimizers.RMSprop(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")


def layer_table(model: keras.Model) -> pd.DataFrame:
    rows = []
    for layer in model.layers:
        rows.append({
            "layer": layer.name,
            "type": layer.__class__.__name__,
            "output_shape": str(getattr(layer, "output", None).shape if hasattr(layer, "output") else ""),
            "params": int(layer.count_params()),
        })
    return pd.DataFrame(rows)


## 4. Predefined Experiment Matrix

The matrix is declared before training. The final ranking uses test accuracy as the primary metric and test loss as the tie-breaker.


In [ ]:
EXPERIMENTS = [
    {"experiment_id": "DNN_01_BASELINE", "task": 1, "model_name": "Dense Baseline", "architecture_key": "dnn_baseline", "optimizer": "adam", "learning_rate": 0.001, "primary_change": "single hidden dense layer"},
    {"experiment_id": "DNN_02_DEEPER", "task": 1, "model_name": "Dense Deeper", "architecture_key": "dnn_deeper", "optimizer": "adam", "learning_rate": 0.001, "primary_change": "adds second dense layer"},
    {"experiment_id": "DNN_03_TUNED", "task": 1, "model_name": "Dense Tuned", "architecture_key": "dnn_tuned", "optimizer": "adam", "learning_rate": 0.001, "primary_change": "adds dropout regularization"},
    {"experiment_id": "CNN_01_BASELINE", "task": 2, "model_name": "CNN Baseline", "architecture_key": "cnn_baseline", "optimizer": "adam", "learning_rate": 0.001, "primary_change": "baseline convolutional model"},
    {"experiment_id": "CNN_02_MORE_FILTERS", "task": 2, "model_name": "CNN More Filters", "architecture_key": "cnn_more_filters", "optimizer": "adam", "learning_rate": 0.001, "primary_change": "increases convolution filters"},
    {"experiment_id": "CNN_03_DEEPER", "task": 2, "model_name": "CNN Deeper", "architecture_key": "cnn_deeper", "optimizer": "adam", "learning_rate": 0.001, "primary_change": "adds convolutional depth"},
    {"experiment_id": "CNN_04_TUNED_FINALIST", "task": 2, "model_name": "CNN Tuned Finalist", "architecture_key": "cnn_tuned_finalist", "optimizer": "adam", "learning_rate": 0.001, "primary_change": "adds batch normalization and dropout"},
    {"experiment_id": "OPT_01_FINAL_CNN_ADAM", "task": 3, "model_name": "Final CNN Adam", "architecture_key": "cnn_tuned_finalist", "optimizer": "adam", "learning_rate": 0.001, "primary_change": "fixed finalist architecture with Adam 0.001"},
    {"experiment_id": "OPT_02_FINAL_CNN_SGD", "task": 3, "model_name": "Final CNN SGD", "architecture_key": "cnn_tuned_finalist", "optimizer": "sgd_momentum", "learning_rate": 0.01, "primary_change": "fixed finalist architecture with SGD momentum"},
    {"experiment_id": "OPT_03_FINAL_CNN_ADAM_LOW_LR", "task": 3, "model_name": "Final CNN Adam Low LR", "architecture_key": "cnn_tuned_finalist", "optimizer": "adam", "learning_rate": 0.0005, "primary_change": "fixed finalist architecture with lower Adam learning rate"},
]

experiment_plan_df = pd.DataFrame(EXPERIMENTS)
display(experiment_plan_df)

for exp in EXPERIMENTS:
    model = build_model(exp["architecture_key"])
    print(f"\n{exp['experiment_id']} | parameters: {model.count_params():,}")
    display(layer_table(model))


## 5. Training and Evaluation

This cell trains all experiments and evaluates each model on the locked test set. Metrics are written to CSV and retained in memory for the visualizations below.


In [ ]:
def run_experiment(exp: dict[str, Any]) -> tuple[dict[str, Any], pd.DataFrame, np.ndarray, np.ndarray]:
    set_seed(SEED)
    model = build_model(exp["architecture_key"], seed=SEED)
    optimizer = build_optimizer(exp["optimizer"], exp["learning_rate"])
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)]
    start = time.perf_counter()
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
    )
    training_time = time.perf_counter() - start

    train_loss, train_accuracy = model.evaluate(x_train, y_train, verbose=0)
    val_loss, val_accuracy = model.evaluate(x_val, y_val, verbose=0)
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
    probabilities = model.predict(x_test, batch_size=BATCH_SIZE, verbose=0)
    predictions = probabilities.argmax(axis=1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_test, predictions, average="macro", zero_division=0)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_test, predictions, average="weighted", zero_division=0)

    record = {
        "experiment_id": exp["experiment_id"],
        "task": exp["task"],
        "model_name": exp["model_name"],
        "architecture_key": exp["architecture_key"],
        "primary_change": exp["primary_change"],
        "optimizer": exp["optimizer"],
        "learning_rate": exp["learning_rate"],
        "batch_size": BATCH_SIZE,
        "epochs_requested": MAX_EPOCHS,
        "epochs_completed": len(history.history["loss"]),
        "early_stopping": f"patience={PATIENCE}, restore_best_weights=True",
        "train_accuracy": float(train_accuracy),
        "validation_accuracy": float(val_accuracy),
        "test_accuracy": float(test_accuracy),
        "train_loss": float(train_loss),
        "validation_loss": float(val_loss),
        "test_loss": float(test_loss),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
        "trainable_params": int(model.count_params()),
        "training_time_seconds": float(training_time),
        "random_seed": SEED,
        "status": "completed",
    }

    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", range(1, len(history_df) + 1))
    history_df.insert(0, "experiment_id", exp["experiment_id"])

    print(f"Completed {exp['experiment_id']}: test_accuracy={test_accuracy:.4f}, test_loss={test_loss:.4f}")
    return record, history_df, predictions, probabilities

records: list[dict[str, Any]] = []
histories: list[pd.DataFrame] = []
predictions_by_experiment: dict[str, np.ndarray] = {}
probabilities_by_experiment: dict[str, np.ndarray] = {}

for experiment in EXPERIMENTS:
    print(f"\n=== Running {experiment['experiment_id']} ===")
    record, history_df, predictions, probabilities = run_experiment(experiment)
    records.append(record)
    histories.append(history_df)
    predictions_by_experiment[experiment["experiment_id"]] = predictions
    probabilities_by_experiment[experiment["experiment_id"]] = probabilities

registry_df = pd.DataFrame(records)
history_all_df = pd.concat(histories, ignore_index=True)

ranking_df = registry_df.sort_values(["test_accuracy", "test_loss"], ascending=[False, True]).reset_index(drop=True)
ranking_df.insert(0, "rank", range(1, len(ranking_df) + 1))

registry_df.to_csv(CSV_DIR / "experiment_registry.csv", index=False)
ranking_df.to_csv(CSV_DIR / "final_ranking.csv", index=False)
history_all_df.to_csv(CSV_DIR / "training_histories.csv", index=False)

best_experiment_id = ranking_df.loc[0, "experiment_id"]
best_predictions = predictions_by_experiment[best_experiment_id]
best_probabilities = probabilities_by_experiment[best_experiment_id]

per_class_report = classification_report(y_test, best_predictions, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
per_class_df = pd.DataFrame(per_class_report).transpose().reset_index().rename(columns={"index": "class_name"})
per_class_df.to_csv(CSV_DIR / "per_class_performance.csv", index=False)

ablation_df = registry_df[["experiment_id", "task", "primary_change", "test_accuracy", "test_loss", "trainable_params", "training_time_seconds"]].copy()
ablation_df.to_csv(CSV_DIR / "ablation_summary.csv", index=False)

print("CSV files written to:", CSV_DIR)
display(ranking_df)
display(per_class_df)


## 6. Ranking and Accuracy Comparison

The official ranking uses test accuracy. Test loss is used only to order ties.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=ranking_df, x="experiment_id", y="test_accuracy", hue="task", dodge=False, ax=ax, palette="viridis")
ax.set_title("Final Model Ranking by Test Accuracy")
ax.set_xlabel("Experiment")
ax.set_ylabel("Test accuracy")
ax.set_ylim(max(0.80, ranking_df["test_accuracy"].min() - 0.02), min(1.0, ranking_df["test_accuracy"].max() + 0.02))
ax.tick_params(axis="x", rotation=55)
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", fontsize=8, padding=2)
plt.tight_layout()
plt.show()

display(Markdown(f"**Best-performing model:** `{best_experiment_id}` with test accuracy `{ranking_df.loc[0, 'test_accuracy']:.4f}` and test loss `{ranking_df.loc[0, 'test_loss']:.4f}`."))


## 7. Learning Curves

These plots show training and validation accuracy/loss from the actual notebook run.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=history_all_df, x="epoch", y="accuracy", hue="experiment_id", marker="o", ax=axes[0])
sns.lineplot(data=history_all_df, x="epoch", y="val_accuracy", hue="experiment_id", marker="o", linestyle="--", ax=axes[0], legend=False)
axes[0].set_title("Training Accuracy (solid) and Validation Accuracy (dashed)")
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend(loc="lower right", fontsize=7)

sns.lineplot(data=history_all_df, x="epoch", y="loss", hue="experiment_id", marker="o", ax=axes[1])
sns.lineplot(data=history_all_df, x="epoch", y="val_loss", hue="experiment_id", marker="o", linestyle="--", ax=axes[1], legend=False)
axes[1].set_title("Training Loss (solid) and Validation Loss (dashed)")
axes[1].set_ylabel("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend(loc="upper right", fontsize=7)
plt.tight_layout()
plt.show()


## 8. Accuracy, Complexity, and Runtime

This comparison checks whether higher accuracy came with materially larger models or longer training time.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=registry_df, x="trainable_params", y="test_accuracy", hue="task", size="training_time_seconds", sizes=(40, 220), ax=axes[0], palette="viridis")
for _, row in registry_df.iterrows():
    axes[0].annotate(row["experiment_id"].replace("_", "\n"), (row["trainable_params"], row["test_accuracy"]), fontsize=6, alpha=0.75)
axes[0].set_title("Accuracy vs Trainable Parameters")
axes[0].set_xlabel("Trainable parameters")
axes[0].set_ylabel("Test accuracy")

sns.barplot(data=registry_df.sort_values("training_time_seconds"), x="experiment_id", y="training_time_seconds", hue="task", dodge=False, ax=axes[1], palette="viridis")
axes[1].set_title("Training Time by Experiment")
axes[1].set_xlabel("Experiment")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=55)
plt.tight_layout()
plt.show()


## 9. Confusion Matrix for Best Model

The confusion matrix is calculated directly from the best model predictions generated in this notebook run.


In [ ]:
cm = confusion_matrix(y_test, best_predictions)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=True, ax=ax)
ax.set_title(f"Confusion Matrix - {best_experiment_id}")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
plt.show()

display(cm_df)


## 10. Misclassification Gallery

The gallery displays examples where the best model was wrong. The confidence shown is the predicted probability for the incorrect predicted class.


In [ ]:
misclassified_indices = np.where(best_predictions != y_test)[0]
if len(misclassified_indices) == 0:
    display(Markdown("No misclassifications were found for the best model."))
else:
    shown = misclassified_indices[:16]
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for ax, idx in zip(axes.ravel(), shown):
        true_label = int(y_test[idx])
        predicted_label = int(best_predictions[idx])
        confidence = float(best_probabilities[idx, predicted_label])
        ax.imshow(x_test[idx].squeeze(), cmap="gray")
        ax.set_title(f"True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[predicted_label]} ({confidence:.2f})", fontsize=8)
        ax.axis("off")
    for ax in axes.ravel()[len(shown):]:
        ax.axis("off")
    fig.suptitle(f"Misclassified Test Images - {best_experiment_id}", fontsize=13)
    plt.tight_layout()
    plt.show()

misclassification_summary = []
for true_id in range(10):
    mask = y_test == true_id
    total = int(mask.sum())
    errors = int((best_predictions[mask] != y_test[mask]).sum())
    misclassification_summary.append({
        "class_id": true_id,
        "class_name": CLASS_NAMES[true_id],
        "test_samples": total,
        "misclassified": errors,
        "class_accuracy": 1 - errors / total,
    })
misclassification_df = pd.DataFrame(misclassification_summary)
misclassification_df.to_csv(CSV_DIR / "misclassification_summary.csv", index=False)
display(misclassification_df.sort_values("class_accuracy"))
